# 01 — Prompt Patterns

Core techniques for effective prompting:
- **Zero-shot**: direct task description, no examples
- **Role-based**: system prompt persona and context
- **Output format control**: reliable JSON output
- **Chain-of-Thought**: step-by-step reasoning
- **Prompt templates**: reusable, parameterized prompts

## 0. Setup

In [ ]:
from openai import OpenAI
import json

# SET YOUR API KEY HERE
api_key = "your-api-key-here"
client = OpenAI(api_key=api_key)

print("✓ Client initialized")

In [ ]:
import os

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

with open(os.path.join(FIXTURES, "tickets.json")) as f:
    tickets = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets")
print("Example:", json.dumps(tickets[0], indent=2))

In [ ]:
def parse_json_safe(text: str) -> dict | None:
    """Parse JSON from LLM response, handling markdown code fences."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print("✓ Helper defined: parse_json_safe")

## 1. Zero-Shot Prompting

The simplest approach — describe the task and ask for the answer directly.

In [ ]:
# Minimal zero-shot
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful customer support assistant."},
        {"role": "user",   "content": (
            "Classify this ticket into one of: billing, technical, account, shipping.\n\n"
            "Ticket: I was charged twice for my subscription this month.\n"
            "Category:"
        )},
    ],
    temperature=0.0,
)
print(repr(response.choices[0].message.content))

In [ ]:
# Problem: output format unpredictable ("billing", "Billing", "The category is billing", etc.)
# Fix: explicit format constraint in system prompt
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Classify support tickets. Reply with ONLY the category name, nothing else."},
        {"role": "user",   "content": (
            "Categories: billing | technical | account | shipping\n\n"
            "Ticket: I was charged twice this month.\n"
            "Category (one word):"
        )},
    ],
    temperature=0.0,
)
print(repr(response.choices[0].message.content))

## 2. Role-Based Prompting (System Prompt)

The system prompt defines the model's **role**, **context**, **valid output values**, and **format**.
This is the most powerful lever in prompt engineering.

In [ ]:
TICKET_SYSTEM_PROMPT = """You are an expert customer support ticket classifier.

Categories:
- billing: payments, charges, refunds, invoices, subscription plans
- technical: bugs, errors, crashes, API failures, performance issues
- account: login, password, profile, permissions, access control
- shipping: delivery, tracking, returns, wrong/missing items

Priorities:
- high: production down, data loss, financial loss, security breach
- medium: impaired functionality, feature broken but workaround exists
- low: questions, minor issues, cosmetic bugs, feature requests

Respond ONLY with valid JSON:
{"category": "billing|technical|account|shipping", "priority": "high|medium|low"}
"""

ticket = tickets[0]["text"]
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": TICKET_SYSTEM_PROMPT},
        {"role": "user",   "content": ticket},
    ],
    temperature=0.0,
)
raw = response.choices[0].message.content
result = parse_json_safe(raw)
print(f"Ticket:   {ticket[:70]}...")
print(f"Raw:      {raw}")
print(f"Parsed:   {result}")
print(f"Expected: category={tickets[0]['category']}, priority={tickets[0]['priority']}")

## 3. Output Format Control

Reliable JSON is critical for downstream processing. Three techniques:

In [ ]:
# Technique 1: JSON schema example in system prompt
SCHEMA_SYSTEM = """You are a ticket classifier. Return this exact JSON structure:
{
    "category": "<billing|technical|account|shipping>",
    "priority": "<high|medium|low>",
    "confidence": <float 0.0-1.0>,
    "summary": "<one sentence>"
}
Output ONLY the JSON, no other text."""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SCHEMA_SYSTEM},
        {"role": "user",   "content": tickets[5]["text"]},
    ],
    temperature=0.0,
)
result = parse_json_safe(response.choices[0].message.content)
print(json.dumps(result, indent=2))

In [ ]:
# Technique 2: response_format={"type": "json_object"} — forces valid JSON output
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": TICKET_SYSTEM_PROMPT},
        {"role": "user",   "content": tickets[3]["text"]},
    ],
    temperature=0.0,
    response_format={"type": "json_object"},  # OpenAI guarantees valid JSON
)
result = json.loads(response.choices[0].message.content)  # no need for parse_json_safe
print(result)

## 4. Chain-of-Thought (CoT) Prompting

Ask the model to reason step-by-step. Improves accuracy on ambiguous tickets.

In [ ]:
COT_SYSTEM = """You are a ticket classifier. Think step by step before classifying.

Steps:
1. What is the core problem?
2. Which category best fits and why?
3. What is the severity/business impact?
4. What priority does this deserve?

Return JSON:
{"reasoning": "<step-by-step analysis>", "category": "...", "priority": "..."}
"""

# Try on an ambiguous ticket that could be billing OR technical
ambiguous_ticket = "The API is returning 500 errors and I'm still being charged for a plan that should have been downgraded."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": COT_SYSTEM},
        {"role": "user",   "content": ambiguous_ticket},
    ],
    temperature=0.0,
)
result = parse_json_safe(response.choices[0].message.content)
print(json.dumps(result, indent=2))

## 5. Prompt Templates

In [ ]:
import re

class PromptTemplate:
    """Reusable parameterized prompt template."""

    def __init__(self, name: str, version: str, system: str, user: str):
        self.name = name
        self.version = version
        self.system = system
        self.user = user

    def format(self, **kwargs) -> tuple[str, str]:
        missing = set(self.variables) - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing variables: {missing}")
        return self.system, self.user.format(**kwargs)

    @property
    def variables(self) -> list[str]:
        return re.findall(r'\{(\w+)\}', self.user)


classify_template = PromptTemplate(
    name="ticket_classifier",
    version="1.0",
    system=TICKET_SYSTEM_PROMPT,
    user="Classify this support ticket:\n\n{ticket}\n\nReturn JSON only.",
)

print(f"Template: {classify_template.name} v{classify_template.version}")
print(f"Variables: {classify_template.variables}")

system, user = classify_template.format(ticket=tickets[0]["text"])
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
    temperature=0.0,
)
print(f"Result: {parse_json_safe(response.choices[0].message.content)}")